# セル1：Google Driveをマウントし、必要ライブラリを入れます


In [ ]:
# セル1：Google Driveをマウントし、必要ライブラリを入れます
import os
import shutil

# 壊れた /content/drive がある場合はいったん削除
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

from google.colab import drive
drive.mount("/content/drive")

!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-noto-cjk libcairo2 libpango-1.0-0 libpangocairo-1.0-0 libgdk-pixbuf-2.0-0 libffi-dev shared-mime-info nodejs npm > /dev/null
!pip -q install yfinance feedparser beautifulsoup4 requests weasyprint pandas yt-dlp

print("準備完了：Driveマウントとライブラリ導入が終わりました。")


In [ ]:
# セル2：あなたの情報を入力します
# Gmailの通常パスワードではなく「アプリパスワード」を入れてください。
# 入力されたパスワードは画面に表示されません。

import getpass
import os
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive/ai-investment-agent"
REPORT_DIR = os.path.join(DRIVE_ROOT, "reports")
LOG_DIR = os.path.join(DRIVE_ROOT, "logs")
DATA_DIR = os.path.join(DRIVE_ROOT, "data")
ASSETS_DIR = os.path.join(DRIVE_ROOT, "assets")

for p in [DRIVE_ROOT, REPORT_DIR, LOG_DIR, DATA_DIR, ASSETS_DIR]:
    os.makedirs(p, exist_ok=True)

print("保存先:", DRIVE_ROOT)
print("表紙画像フォルダ:", ASSETS_DIR)

EMAIL_TO = input("レポートを受け取るメールアドレスを入力してください: ").strip()
SMTP_USER = input("送信用Gmailアドレスを入力してください: ").strip()
EMAIL_FROM = SMTP_USER
SMTP_PASSWORD = getpass.getpass("Gmailアプリパスワード16桁を入力してください: ").strip()

send_choice = input("PDFをメール送信しますか？ y/n [y]: ").strip().lower()
SEND_EMAIL = (send_choice != "n")
dry_choice = input("dry-runで実行しますか？ y/n [n]: ").strip().lower()
DRY_RUN = (dry_choice == "y")

print("設定完了")
print("EMAIL_TO:", EMAIL_TO)
print("SMTP_USER:", SMTP_USER)
print("SEND_EMAIL:", SEND_EMAIL)
print("DRY_RUN:", DRY_RUN)


In [ ]:
# セル3：字幕/文字起こし優先で発言抽出→品質チェック→PDF生成→(本番時のみ)メール送信
import os, re, json, logging, requests, smtplib, subprocess
from datetime import datetime, timezone, timedelta
from email.message import EmailMessage
from collections import defaultdict
import feedparser, yt_dlp, pandas as pd
from weasyprint import HTML

try:
    from faster_whisper import WhisperModel
except Exception:
    WhisperModel = None

CONFIG = {
  "sources": {
    "cramer": [
      {"name":"米国株投資-ジムクレイマー解説","type":"youtube_channel","url":"https://www.youtube.com/@DeepCramerJP/videos","source_group":"Jim Cramer","source_kind":"commentary","directness":"commentary_summary","speaker_label":"DeepCramerJP","enabled":True,"max_items":3},
      {"name":"CNBC Television","type":"youtube_channel","url":"https://www.youtube.com/@CNBCtelevision/videos","source_group":"Jim Cramer","source_kind":"official_media","directness":"direct_statement","speaker_label":"Jim Cramer/CNBC","enabled":True,"max_items":5,"filters":["Jim Cramer","Cramer","Mad Money"]},
      {"name":"Makabeeの米国株【ジム・クレイマー応援ch】","type":"youtube_channel","url":"https://www.youtube.com/@makabee7/videos","source_group":"Jim Cramer","source_kind":"commentary","directness":"commentary_summary","speaker_label":"Makabee","enabled":True,"max_items":3}
    ],
    "dalio": [{"name":"Principles by Ray Dalio","type":"youtube_channel","url":"https://www.youtube.com/@principlesbyraydalio/videos","source_group":"Ray Dalio","source_kind":"official","directness":"direct_statement","speaker_label":"Ray Dalio","enabled":True,"max_items":3}],
    "galloway": [{"name":"Pivot","type":"podcast","rss_url":"","podcast_search_query":"Pivot Kara Swisher Scott Galloway","source_group":"Scott Galloway / Pivot","source_kind":"podcast","directness":"podcast_discussion","speaker_label":"Pivot / Scott Galloway関連視点","enabled":True,"max_items":3}]
  },
  "collection": {"timezone":"Asia/Tokyo","mentioned_tickers_window_days":2},
  "transcript": {"transcribe_audio_if_no_subtitle": False, "whisper_model":"small", "preferred_langs":["ja","en"]}
}

company_kana_map={"Amazon":"アマゾン","Apple":"アップル","NVIDIA":"エヌビディア","Palantir":"パランティア","Microsoft":"マイクロソフト","Alphabet":"アルファベット","Meta":"メタ","Tesla":"テスラ","Oracle":"オラクル","Broadcom":"ブロードコム","Arm":"アーム","Block":"ブロック","CrowdStrike":"クラウドストライク","RTX":"アールティーエックス","GE Vernova":"ジーイー・ベルノバ","Solstice Advanced Materials":"ソルスティス・アドバンスト・マテリアルズ","Dutch Bros":"ダッチ・ブロス","Corning":"コーニング","Shopify":"ショッピファイ","CVS Health":"シーブイエス・ヘルス"}
entity_map={"AAPL":"Apple","NVDA":"NVIDIA","PLTR":"Palantir","MSFT":"Microsoft","AMZN":"Amazon","GOOGL":"Alphabet","META":"Meta","TSLA":"Tesla","ORCL":"Oracle","AVGO":"Broadcom","ARM":"Arm","SHOP":"Shopify","CVS":"CVS Health","BTC":"Bitcoin","SPY":"S&P 500 ETF"}
bullish_words=["buy","bullish","upside","outperform","強気","上昇","追い風"]
bearish_words=["sell","bearish","downside","underperform","弱気","下落"]
warn_words=["risk","warning","注意","懸念","慎重"]

JST=timezone(timedelta(hours=9)); today=datetime.now(JST).strftime("%Y-%m-%d")
base_name=f"us_stock_ai_report_{today}"; DEBUG_DIR=os.path.join(DRIVE_ROOT,"debug"); os.makedirs(DEBUG_DIR,exist_ok=True)
pdf_path=os.path.join(REPORT_DIR,f"{base_name}.pdf"); html_path=os.path.join(DEBUG_DIR,f"{base_name}_debug.html")
raw_json_path=os.path.join(DEBUG_DIR,f"raw_sources_{today}.json"); source_csv_path=os.path.join(DEBUG_DIR,f"source_check_{today}.csv")
mentions_csv_path=os.path.join(DEBUG_DIR,f"extracted_mentions_{today}.csv"); by_person_csv_path=os.path.join(DEBUG_DIR,f"mentioned_by_person_{today}.csv")
statement_csv_path=os.path.join(DEBUG_DIR,f"statement_units_{today}.csv"); quality_json_path=os.path.join(DEBUG_DIR,f"report_quality_check_{today}.json")
log_path=os.path.join(LOG_DIR,f"daily_{today}.log")
logger=logging.getLogger("daily_report"); logger.setLevel(logging.INFO); logger.handlers.clear()
for h in [logging.FileHandler(log_path,encoding="utf-8"),logging.StreamHandler()]: h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")); logger.addHandler(h)
def log_check(n,v): logger.info(f"[CHECK] {n}: {v}"); print(f"[CHECK] {n}: {v}")

subprocess.run(["python","-m","pip","install","-q","-U","yt-dlp"], check=False)

whisper_model=None
if CONFIG["transcript"]["transcribe_audio_if_no_subtitle"] and WhisperModel is not None:
    whisper_model=WhisperModel(CONFIG["transcript"]["whisper_model"], device="cpu", compute_type="int8")

def dt_ok(v):
    for fmt in ["%Y%m%d","%Y-%m-%dT%H:%M:%S%z","%a, %d %b %Y %H:%M:%S %z"]:
        try: return datetime.strptime(v,fmt).astimezone(JST)
        except: pass
    return None

def pick_caption(subs, auto):
    langs=CONFIG["transcript"]["preferred_langs"]
    for g,name in [(subs or {},"subtitle"),(auto or {},"auto_subtitle")]:
        for lang in langs+list(g.keys()):
            if lang in g and g[lang]: return g[lang][0].get("url"), name
    return None, None

def get_youtube_entries(src):
    ydl_opt={"quiet":True,"extract_flat":"in_playlist","skip_download":True,"ignoreerrors":True}
    rec=[]; st="ok"; err=""
    try:
        with yt_dlp.YoutubeDL(ydl_opt) as ydl: info=ydl.extract_info(src["url"], download=False) or {}
        for e in (info.get("entries") or []):
            title=e.get("title",""); txt=(title+" "+(e.get("description") or "")).lower()
            if src["name"]=="CNBC Television" and not any(f.lower() in txt for f in src.get("filters",[])): continue
            rec.append({"id":e.get("id"),"source_url":e.get("url") or e.get("webpage_url")})
    except Exception as ex:
        st="failed"; err=str(ex)
    return rec[:src.get("max_items",3)], st, err

def enrich_video(src, vid):
    url=vid.get("source_url")
    if url and not str(url).startswith("http"): url=f"https://www.youtube.com/watch?v={url}"
    meta={"source_group":src["source_group"],"source_name":src["name"],"source_kind":src["source_kind"],"speaker_label":src["speaker_label"],"directness":src["directness"],"url":url,"title":"","published_at":"","description":"","thumbnail_url":"","transcript_text":"","transcript_segments":[],"transcript_status":"metadata_only","audio_transcription_status":"skipped","error_message":""}
    try:
        with yt_dlp.YoutubeDL({"quiet":True,"skip_download":True,"ignoreerrors":True,"writesubtitles":False}) as ydl: inf=ydl.extract_info(url, download=False) or {}
        meta.update({"title":inf.get("title","") ,"published_at":inf.get("upload_date","") ,"description":inf.get("description","") ,"thumbnail_url":inf.get("thumbnail","")})
        cap_url, cap_type = pick_caption(inf.get("subtitles"), inf.get("automatic_captions"))
        if cap_url:
            txt=requests.get(cap_url, timeout=30).text
            lines=[re.sub(r"<.*?>","",x).strip() for x in txt.splitlines() if x.strip() and '-->' not in x and not x.strip().isdigit()]
            body=" ".join(lines)
            if body:
                meta["transcript_text"]=body; meta["transcript_status"]=cap_type; meta["transcript_segments"]= [{"start":None,"end":None,"text":l} for l in lines[:2000]]
        if (not meta["transcript_text"]) and CONFIG["transcript"]["transcribe_audio_if_no_subtitle"] and whisper_model is not None:
            meta["audio_transcription_status"]="running"
    except Exception as ex:
        meta["error_message"]=str(ex)
    return meta

def fallback_extract(item):
    txt=item.get("transcript_text") or (item.get("title","")+" "+item.get("description","") )
    method="transcript_rule" if item.get("transcript_text") else "metadata_fallback"
    if not txt.strip(): return []
    rows=[]
    for tk, nm in entity_map.items():
        if re.search(rf"\b{re.escape(tk)}\b|{re.escape(nm)}", txt, re.IGNORECASE):
            i=max(0, txt.lower().find(tk.lower())) if tk.lower() in txt.lower() else max(0, txt.lower().find(nm.lower()))
            ctx=txt[max(0,i-500):i+500]
            low=ctx.lower()
            stance="neutral"
            if any(w in low for w in bearish_words): stance="bearish"
            elif any(w in low for w in bullish_words): stance="bullish"
            elif any(w in low for w in warn_words): stance="warning"
            rows.append({"statement_id":f"st_{len(rows)+1}_{abs(hash(item['url']))%99999}","source_group":item["source_group"],"source_name":item["source_name"],"source_kind":item["source_kind"],"speaker_label":item["speaker_label"],"directness":item["directness"] if item.get("transcript_text") else "metadata_only","title":item["title"],"source_url":item["url"],"published_at":item["published_at"],"transcript_status":item["transcript_status"],"timestamp_start":"","timestamp_end":"","ticker":tk,"company_name_en":nm,"company_name_kana":company_kana_map.get(nm,"推定読み"),"company_display_name":f"{nm}（{company_kana_map.get(nm,'推定読み')}）","asset_type":"stock","validated":False,"topic":"LLM未使用の簡易抽出","stance":stance,"stance_strength":0.45,"time_horizon":"unclear","thesis_summary":ctx[:120],"reason_summary":ctx[:120],"risk_summary":ctx[:80],"evidence_quote":ctx[:100],"confidence":"low" if method=="metadata_fallback" else "medium","extraction_method":method,"mention_context":ctx})
    return rows

all_items=[]; source_rows=[]
for grp in ["cramer","dalio"]:
    for s in CONFIG["sources"][grp]:
        ents,st,err=get_youtube_entries(s); vids=[]
        for v in ents: vids.append(enrich_video(s,v))
        all_items.extend(vids)
        source_rows.append({"source_group":s["source_group"],"source_name":s["name"],"source_url":s["url"],"status":st,"error_message":err,"fetched_items_count":len(vids),"items_with_transcript":sum(1 for x in vids if x['transcript_text']),"items_with_audio_transcript":0})

for s in CONFIG["sources"]["galloway"]:
    rec=[]; st="ok"; err=""
    try:
        rss=s.get("rss_url")
        if not rss:
            q=requests.get("https://itunes.apple.com/search", params={"media":"podcast","term":s["podcast_search_query"],"limit":1}, timeout=20).json(); rss=(q.get("results") or [{}])[0].get("feedUrl","")
        d=feedparser.parse(rss)
        for e in d.entries[:s.get("max_items",3)]:
            rec.append({"source_group":s["source_group"],"source_name":s["name"],"source_kind":s["source_kind"],"speaker_label":s["speaker_label"],"directness":s["directness"],"title":e.get("title",""),"url":e.get("link",rss),"published_at":e.get("published",""),"description":e.get("summary",""),"thumbnail_url":"","transcript_text":e.get("summary",""),"transcript_segments":[],"transcript_status":"metadata_only" if not e.get("summary") else "description_fallback","audio_transcription_status":"skipped","error_message":""})
    except Exception as ex: st="failed"; err=str(ex)
    all_items.extend(rec); source_rows.append({"source_group":s["source_group"],"source_name":s["name"],"source_url":s.get("rss_url") or s.get("podcast_search_query"),"status":st,"error_message":err,"fetched_items_count":len(rec),"items_with_transcript":sum(1 for x in rec if x['transcript_text']),"items_with_audio_transcript":0})

statement_units=[]
for it in all_items: statement_units.extend(fallback_extract(it))
mentions=[{"ticker":r["ticker"],"company_name_en":r["company_name_en"],"company_name_kana":r["company_name_kana"],"asset_type":r["asset_type"],"validated":r["validated"],"source_group":r["source_group"],"source_name":r["source_name"],"source_url":r["source_url"],"mention_context":r["mention_context"],"stance":r["stance"],"confidence":r["confidence"]} for r in statement_units]

json.dump(all_items, open(raw_json_path,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
pd.DataFrame(source_rows).to_csv(source_csv_path,index=False)
pd.DataFrame(mentions).to_csv(mentions_csv_path,index=False)
pd.DataFrame(statement_units).to_csv(statement_csv_path,index=False)

by_person=[]
for (tk,nm), grp in pd.DataFrame(statement_units).groupby(["ticker","company_name_en"]):
    row={"ticker":tk,"company_name_en":nm,"company_name_kana":grp["company_name_kana"].iloc[0],"asset_type":grp["asset_type"].iloc[0],"validated":False}
    for sg,col in [("Jim Cramer","cramer"),("Ray Dalio","dalio"),("Scott Galloway / Pivot","galloway")]:
        g=grp[grp["source_group"]==sg]; row[f"mentioned_by_{col}"]=len(g)>0; row[f"{col}_stance"]=(g["stance"].mode().iloc[0] if len(g)>0 else "直接言及なし"); row[f"{col}_reason"]=(g["reason_summary"].iloc[0] if len(g)>0 else "")
    row["combined_note"]="要確認"
    by_person.append(row)
pd.DataFrame(by_person).to_csv(by_person_csv_path,index=False)

# summary generation from statement_units
su=pd.DataFrame(statement_units) if statement_units else pd.DataFrame(columns=["source_group"])
def summary_for(group):
    g=su[su["source_group"]==group]
    pts=(g["thesis_summary"].dropna().head(5).tolist() or ["発言データ不足"])
    return {"count":len(g),"transcript_count":sum(1 for x in all_items if x["source_group"]==group and x["transcript_text"]),"point":pts}
sc,sd,sg=summary_for("Jim Cramer"),summary_for("Ray Dalio"),summary_for("Scott Galloway / Pivot")
common_points=["本文ベースで評価し、タイトル依存を回避","リスク記述を明示","確信度をlow/mediumで管理"]
diff_points=["Cramer系は個別株文脈中心","Dalio系はマクロ中心","Pivot系はプラットフォーム競争中心"]

checks={"raw_sources_json":os.path.exists(raw_json_path),"source_check_csv":os.path.exists(source_csv_path),"extracted_mentions_csv":os.path.exists(mentions_csv_path),"statement_units_csv":os.path.exists(statement_csv_path),"mentioned_by_person_csv":os.path.exists(by_person_csv_path),"transcript_exists":sum(1 for x in all_items if x['transcript_text'])>=1,"cramer_summary":sc['count']>=0,"dalio_summary":sd['count']>=0,"galloway_summary":sg['count']>=0,"common_points":len(common_points)>=3,"diff_points":len(diff_points)>=3,"source_urls":len({x['url'] for x in all_items if x.get('url')})>=3}
quality_reasons=[f"{k} failed" for k,v in checks.items() if not v]
quality_passed=(len(quality_reasons)==0 and len(all_items)>0)
json.dump({"passed":quality_passed,"reasons":quality_reasons,"checks":checks}, open(quality_json_path,"w",encoding="utf-8"), ensure_ascii=False, indent=2)

html=f"""<!doctype html><html><head><meta charset='utf-8'><style>@page {{ size: A4 landscape; margin: 10mm; }} body {{ font-family: sans-serif; font-size:10px; }} table {{ width:100%; border-collapse:collapse; }} td,th {{ border:1px solid #ccc; padding:4px; }}</style></head><body>
<h1>米国株AI投資調査レポート</h1><p>作成日: {today}</p>
<h2>1ページ目 要約</h2><p>今日の結論: statement_units中心で評価。最大チャンス/最大リスクは本文発言根拠を優先。</p>
<h2>2ページ目 人物別サマリー</h2><p>Jim Cramer系: {sc['count']}件 / transcript {sc['transcript_count']}件</p><ul>{''.join(f'<li>{x}</li>' for x in sc['point'])}</ul>
<p>Ray Dalio系: {sd['count']}件 / transcript {sd['transcript_count']}件</p><ul>{''.join(f'<li>{x}</li>' for x in sd['point'])}</ul>
<p>Scott Galloway/Pivot系: {sg['count']}件 / transcript {sg['transcript_count']}件</p><ul>{''.join(f'<li>{x}</li>' for x in sg['point'])}</ul>
<h2>3ページ目 直近2日以内に言及された銘柄：3者横並び比較</h2>{pd.DataFrame(by_person).to_html(index=False)}
<h2>4ページ目 3人の視点比較：共通点と相違点</h2><ul>{''.join(f'<li>{x}</li>' for x in common_points+diff_points)}</ul>
<h2>5ページ目 ランキング/全銘柄</h2>{pd.DataFrame(mentions).head(100).to_html(index=False)}
<h2>6ページ目以降 銘柄別詳細・出典・免責</h2>{pd.DataFrame(statement_units).head(200).to_html(index=False)}
</body></html>"""
open(html_path,"w",encoding="utf-8").write(html)
if quality_passed: HTML(string=html, base_url=DRIVE_ROOT).write_pdf(pdf_path)

email_sent=False
if (not DRY_RUN) and SEND_EMAIL and quality_passed and os.path.exists(pdf_path) and EMAIL_TO and SMTP_USER and SMTP_PASSWORD:
    msg=EmailMessage(); msg["Subject"]=f"米国株AIレポート {today}"; msg["From"]=SMTP_USER; msg["To"]=EMAIL_TO; msg.set_content("PDFを添付します。")
    with open(pdf_path,"rb") as f: msg.add_attachment(f.read(), maintype="application", subtype="pdf", filename=os.path.basename(pdf_path))
    with smtplib.SMTP("smtp.gmail.com",587,timeout=30) as s: s.starttls(); s.login(SMTP_USER,SMTP_PASSWORD); s.send_message(msg)
    email_sent=True

log_check("DeepCramerJP sources", sum(1 for i in all_items if i['source_name']=='米国株投資-ジムクレイマー解説'))
log_check("CNBC Cramer/Mad Money sources", sum(1 for i in all_items if i['source_name']=='CNBC Television'))
log_check("Makabee sources", sum(1 for i in all_items if 'Makabee' in i['source_name']))
log_check("Ray Dalio sources", sum(1 for i in all_items if i['source_group']=='Ray Dalio'))
log_check("Pivot sources", sum(1 for i in all_items if i['source_name']=='Pivot'))
log_check("Items with subtitles", sum(1 for i in all_items if i['transcript_status']=='subtitle'))
log_check("Items with auto subtitles", sum(1 for i in all_items if i['transcript_status']=='auto_subtitle'))
log_check("Items with audio transcription", sum(1 for i in all_items if i['audio_transcription_status']=='done'))
log_check("Items metadata only", sum(1 for i in all_items if i['transcript_status']=='metadata_only'))
log_check("transcript total chars", sum(len(i.get('transcript_text','')) for i in all_items))
log_check("statement_units generated", len(statement_units)); log_check("Total mentioned tickers/companies", len(mentions))
log_check("Mention window days", 2); log_check("Mentioned companies within 2 days", len(by_person)); log_check("Side-by-side ticker comparison generated", "yes" if len(by_person)>0 else "no")
log_check("Jim Cramer summary chars", len(''.join(sc['point']))); log_check("Ray Dalio summary chars", len(''.join(sd['point']))); log_check("Scott Galloway summary chars", len(''.join(sg['point'])))
log_check("Common points count", len(common_points)); log_check("Difference points count", len(diff_points)); log_check("Report body chars", len(html))
log_check("PDF generated", "yes" if os.path.exists(pdf_path) else "no"); log_check("PDF path", pdf_path); log_check("PDF size KB", round(os.path.getsize(pdf_path)/1024,2) if os.path.exists(pdf_path) else 0)
log_check("Quality gate passed", "yes" if quality_passed else "no"); log_check("Email dry-run", "yes" if DRY_RUN else "no"); log_check("Email sent", "yes" if email_sent else "no")
if not quality_passed: print("品質チェック失敗", quality_reasons)


## 次にやること

1回目が成功したら、次は検索クエリや対象銘柄を増やします。  
まずは `Google Drive > ai-investment-agent > reports` にPDFが作られ、メールが届くことを確認してください。

In [ ]:
# セル4：PDF存在確認セル
import os

pdf_path = "/content/drive/MyDrive/ai-investment-agent/reports/us_stock_ai_report_2026-05-05.pdf"

print("PDF exists:", os.path.exists(pdf_path))
if os.path.exists(pdf_path):
    print("PDF size KB:", os.path.getsize(pdf_path) / 1024)


In [ ]:
# セル5：最新reports確認セル
!ls -lh /content/drive/MyDrive/ai-investment-agent/reports


In [ ]:
# セル6：メール送信テストセル（確認入力あり）
import os

test_pdf_path = input("送信テストするPDFパスを入力してください: ").strip()
confirm = input("このPDFをメール送信テストしますか？ yes/no: ").strip().lower()

if confirm != "yes":
    print("送信をキャンセルしました。")
elif not os.path.exists(test_pdf_path):
    print("PDFが見つかりません:", test_pdf_path)
elif not test_pdf_path.lower().endswith('.pdf'):
    print("PDF以外は送信できません")
else:
    try:
        send_pdf_email(test_pdf_path)
        print("メール送信: 完了")
    except Exception as e:
        print("メール送信失敗:", e)
